# Predicting Aerobic Fitness from Wearable Sensor Data

Final year project implementation — Rahman Rufai-Alao (S122202065).

The data (De Brabandere et al., 2018, PLOS ONE) has 41 treadmill runs with heart-rate and
body-worn accelerometer signals plus a lab VO2max for each runner. I turn VO2max into a
Low/High fitness label and compare Logistic Regression, Random Forest and SVM on the sensor
features. At the end there is a small demographic estimator that anyone can try.


In [ ]:
!pip install -q gradio

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             RocCurveDisplay, classification_report)

SEED = 42
np.random.seed(SEED)

## 1. Load the data

In [ ]:
base = "https://ndownloader.figshare.com/files/"
subj  = pd.read_csv(base + "11969639")   # subject info + VO2max
feats = pd.read_csv(base + "11969642")   # the 490 sensor features

# the csv headers have leading spaces, tidy them up
for df in (subj, feats):
    df.columns = df.columns.str.strip()

print(subj.shape, feats.shape)
subj.head()

**Note on sample size:** 31 unique participants contributed a total of 41 submaximal runs — most completed one test, and some completed both a *pre* and *post* session (see the `Test` column). This is why the merged dataset used for modelling below has 41 rows rather than 31.

## 1b. Data cleaning & outlier screening

The heart-rate and accelerometer signals distributed here are already pre-extracted into candidate features by De Brabandere et al. (2018), rather than raw time-series, so the per-sample smoothing described in the methodology was performed upstream by their extraction pipeline. This section applies the outlier-screening step (Section 3.6.2) to the demographic and VO2max fields: records with |z| > 3 on any of these variables are flagged for review rather than dropped automatically, given the small sample.

In [ ]:
from scipy.stats import zscore

check_cols = [c for c in subj.select_dtypes("number").columns if c != "Subject ID"]
z_scores = subj[check_cols].apply(zscore)
outlier_mask = (z_scores.abs() > 3).any(axis=1)

print(f"Flagged {outlier_mask.sum()} potential outlier record(s) (|z| > 3) out of {len(subj)} total.")
if outlier_mask.any():
    display(subj.loc[outlier_mask, check_cols])
else:
    print("None flagged — given the small sample, all records are retained as-is.")

## 2. Explore VO2max

In [ ]:
vo2 = [c for c in subj.columns if "VO2" in c][0]
print("target:", vo2)

plt.hist(subj[vo2], bins=10, edgecolor="black")
plt.axvline(subj[vo2].median(), color="red", ls="--", label="median")
plt.xlabel("VO2max (ml/kg/min)"); plt.ylabel("runners"); plt.legend()
plt.title("VO2max distribution"); plt.show()

subj.describe().round(2)

## 2b. Explore the wearable sensor features

Objective 1 also calls for exploring the heart-rate and accelerometer variables themselves, not just VO2max. The features file has 490 candidate columns; below are the 15 most strongly correlated with VO2max, giving an early sense of which signals carry the most information before formal feature selection in Section 5.

In [ ]:
merge_keys = [c for c in feats.columns if c in subj.columns]
sensor_view = feats.merge(subj[merge_keys + [vo2]], on=merge_keys)

numeric_feats = sensor_view.drop(columns=merge_keys + [vo2]).select_dtypes("number")
corrs = numeric_feats.corrwith(sensor_view[vo2]).sort_values(key=abs, ascending=False)

top15 = corrs.head(15)
plt.figure(figsize=(8, 5))
colors = ["#2a9d8f" if v > 0 else "#e76f51" for v in top15[::-1]]
top15[::-1].plot(kind="barh", color=colors)
plt.xlabel("Correlation with VO2max")
plt.title("Top 15 sensor features most correlated with VO2max")
plt.tight_layout(); plt.show()

top15

## 3. Make the fitness label

Split VO2max at the median so the two classes stay balanced (only 41 runners).

In [ ]:
subj["fitness"] = (subj[vo2] >= subj[vo2].median()).astype(int)   # 1 = High, 0 = Low
subj["fitness"].value_counts()

## 4. Build the feature matrix

Each feature row is matched to its subject on the Subject ID + Test key so no row gets
duplicated. If the key is not unique the rows are lined up by order instead.

In [ ]:
keys = [c for c in feats.columns if c in subj.columns]
data = feats.merge(subj[keys + ["fitness"]], on=keys)

if len(data) != len(feats):
    data = feats.copy()
    data["fitness"] = subj["fitness"].values

drop = keys + ["fitness"] + [c for c in data.columns if "VO2" in c]
X = data.drop(columns=[c for c in drop if c in data.columns]).select_dtypes("number")
y = data["fitness"].values

print("X:", X.shape, "| classes:", np.bincount(y))

## 5. The three models

Each is a pipeline: scale -> keep the 8 most useful features -> classify. Feature selection
lives inside the pipeline so it is redone in every fold and never peeks at the test data.
No neural net here — 41 rows is far too little for one.

In [ ]:
def pipe(model, k=8):
    return Pipeline([("scale", MinMaxScaler()),
                     ("pick", SelectKBest(f_classif, k=k)),
                     ("model", model)])

models = {
    "Logistic Regression": pipe(LogisticRegression(max_iter=1000)),
    "Random Forest": pipe(RandomForestClassifier(n_estimators=300, random_state=SEED)),
    "SVM": pipe(SVC(kernel="rbf", probability=True, random_state=SEED)),
}
list(models)

## 6. Cross-validation

Stratified 5-fold, reported as mean ± std across the folds.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]

rows = {}
for name, p in models.items():
    s = cross_validate(p, X, y, cv=cv, scoring=metrics)
    rows[name] = {m: f"{s['test_'+m].mean():.3f} ± {s['test_'+m].std():.3f}" for m in metrics}

pd.DataFrame(rows).T

## 7. A closer look at the best model

In [ ]:
f1 = {n: cross_validate(p, X, y, cv=cv, scoring="f1")["test_score"].mean()
      for n, p in models.items()}
best = max(f1, key=f1.get)
print("best model:", best)

bp = models[best]
pred = cross_val_predict(bp, X, y, cv=cv)
prob = cross_val_predict(bp, X, y, cv=cv, method="predict_proba")[:, 1]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(y, pred),
                       display_labels=["Low", "High"]).plot(ax=ax[0], colorbar=False)
RocCurveDisplay.from_predictions(y, prob, ax=ax[1])
plt.tight_layout(); plt.show()

print(classification_report(y, pred, target_names=["Low", "High"]))

# Feature importance, if the best model is Random Forest
if best == "Random Forest":
    bp.fit(X, y)
    selected_idx = bp.named_steps["pick"].get_support(indices=True)
    selected_names = X.columns[selected_idx]
    importances = pd.Series(bp.named_steps["model"].feature_importances_,
                             index=selected_names).sort_values()

    plt.figure(figsize=(7, 4))
    importances.plot(kind="barh", color="#2a9d8f")
    plt.xlabel("Feature importance")
    plt.title("Random Forest feature importance (selected features)")
    plt.tight_layout(); plt.show()

## 8. Fitness checker app (Gradio)

A version an individual can actually use with a fitness watch. You enter your age, body mass
and your heart rate during an easy warm-up and while running. Heart rate is the wearable part
here: a fitter person's heart beats slower for the same effort, so those two numbers carry
most of the signal. It replies with one clear line.

In [ ]:
import gradio as gr

age_c  = next(c for c in subj.columns if c.startswith("Age"))
mass_c = next(c for c in subj.columns if "Body mass" in c)

# line each runner's profile up with their heart-rate readings
keys = [c for c in feats.columns if c in subj.columns]
app  = subj[keys + [age_c, mass_c, "fitness"]].merge(feats[keys + ["HR_0", "HR_2"]], on=keys)

Xa = app[[age_c, mass_c, "HR_0", "HR_2"]].astype(float)     # HR_0 = warm-up HR, HR_2 = running HR
checker = Pipeline([("scale", MinMaxScaler()),
                    ("rf", RandomForestClassifier(n_estimators=300, random_state=SEED))])
checker.fit(Xa, app["fitness"])

labels = ["Age (years)", "Body mass (kg)",
          "Heart rate during warm-up (bpm)", "Heart rate while running (bpm)"]
mid = Xa.median().round(0)

def check(age, mass, hr_warm, hr_run):
    row = pd.DataFrame([[age, mass, hr_warm, hr_run]], columns=Xa.columns)
    p = checker.predict_proba(row)[0]
    verdict = "HIGH" if p[1] >= 0.5 else "LOW"
    return f"{verdict} aerobic fitness  ({round(max(p) * 100)}% confidence)"

boxes = [gr.Number(label=labels[i], value=float(mid.iloc[i])) for i in range(4)]
gr.Interface(check, boxes, gr.Textbox(label="Result"),
             title="Aerobic Fitness Checker",
             description="Enter your age, body mass and heart rate (bpm) during an easy "
                         "warm-up and while running.").launch(share=True)